In [1]:
"""
CUSTOMER CHURN PREDICTION — END-TO-END PROJECT
Dataset expected: Bank Customer Churn Prediction.csv
Target: churn (0 = stayed, 1 = churned)

Run:
    pip install pandas numpy matplotlib seaborn scikit-learn joblib
    python customer_churn_project.py

This script keeps learned feature engineering and preprocessing inside the
cross-validation pipeline to prevent train/test leakage.
"""

from pathlib import Path
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier,
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    RocCurveDisplay,
    PrecisionRecallDisplay,
)
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
    GridSearchCV,
)
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
pd.set_option("display.max_columns", 100)

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------
DATA_PATH = Path("Bank Customer Churn Prediction.csv")
OUTPUT_DIR = Path("churn_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

TARGET = "churn"
ID_COLUMN = "customer_id"
RANDOM_STATE = 42
TEST_SIZE = 0.20
CV_SPLITS = 5

# ------------------------------------------------------------
# 2. LOAD DATA
# ------------------------------------------------------------
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH.resolve()}. "
        "Place the CSV in the same folder as this script or update DATA_PATH."
    )

df = pd.read_csv(DATA_PATH)

required_columns = {
    "customer_id", "credit_score", "country", "gender", "age", "tenure",
    "balance", "products_number", "credit_card", "active_member",
    "estimated_salary", "churn"
}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f"Dataset is missing columns: {sorted(missing_columns)}")

print("\n1) DATASET OVERVIEW")
print("Shape:", df.shape)
print(df.head())
print("\nData types and non-null counts:")
df.info()
print("\nMissing values:")
print(df.isna().sum())


1) DATASET OVERVIEW
Shape: (10000, 12)
   customer_id  credit_score country  gender  age  tenure    balance  \
0     15634602           619  France  Female   42       2       0.00   
1     15647311           608   Spain  Female   41       1   83807.86   
2     15619304           502  France  Female   42       8  159660.80   
3     15701354           699  France  Female   39       1       0.00   
4     15737888           850   Spain  Female   43       2  125510.82   

   products_number  credit_card  active_member  estimated_salary  churn  
0                1            1              1         101348.88      1  
1                1            0              1         112542.58      0  
2                3            1              0         113931.57      1  
3                2            0              0          93826.63      0  
4                1            1              1          79084.10      0  

Data types and non-null counts:
<class 'pandas.DataFrame'>
RangeIndex: 10000 entri